# Per-order RV scan — 0 → 1 km/s in 100 m/s steps

Same physical path and calibrations as `rv_recovery_sim.ipynb` (spectrum Doppler
paint, near-noiseless UTR, shared Aidan+ZP₀ WAVE / fiberflat, sky_sub and barycorr off,
line-by-line `ccfopt` stellar RV), but steps the injected RV from **0** to
**+1000 m/s** in **100 m/s** increments (11 runs) and records the recovered RV
**per echelle order** from each L2 `RVCCF_FIB*` table.

Workflow (each stage is independently re-runnable):

1. Sync KOA caldb; configure fiber / mask / CCF kwargs + cache flags.
2. Build (or reload) shared Stage-1 cals at RV=0.
3. **Raw sim cache** — for every inject step, run
   1D → Doppler → ZEMAX → UTR once and save `simulated_ramp.fits` /
   `rate_map.fits`. With `RESIMULATE_RAW=False` (default), existing ramps are
   reused and this step is a no-op.
4. **Pipeline (no stellar RV)** — Stage0→1→2 through blaze (+ sky_sub and barycorr off);
   `stellar_rv` is skipped. With `RERUN_PIPELINE=False` (default), existing
   blaze-corrected L2 products are reused.
5. **Stellar RV only** — run `StellarRVPrimitive` on each blaze L2, write
   `RVCCF_FIB*` / `RVCCFAVG_FIB*`, and build `per_order_rv_scan.ecsv`.
   Set `RERUN_STELLAR_RV=True` to refresh all injects after mask / CCF changes
   without re-simulating or re-reducing.
6. Plot from `per_order_rv_scan.ecsv` (disk only — no pipeline required).

Uses the packaged static `*.ccfopt.delta.mas` (no re-zero during the scan).


In [ ]:
import os
import hispecdrp

print(f"hispecdrp {hispecdrp.__version__} @ {hispecdrp.__file__}")

OUT = "/Users/ckilpatrick/hispec_rv_order_scan"
os.makedirs(OUT, exist_ok=True)

os.environ.setdefault(
    "KOA_CALIBRATION_CACHE",
    os.path.join(OUT, "cals", "KOA_CALIBRATION_CACHE"),
)
print("output:", OUT)
print("KOA_CALIBRATION_CACHE:", os.environ["KOA_CALIBRATION_CACHE"])


## 1. Sync / verify KOA caldb (BSPEC)

Uses `ensure_koa_calibrations`: seed the notebook cache from another populated
KOA cache if needed, try remote sync, and fall back to local-only on auth /
network / API failures.


In [ ]:
from hispecdrp.calibrations import ensure_koa_calibrations

SPECTROGRAPH = "BSPEC"

# Optional: pin an explicit seed cache (otherwise auto-discovers
# ~/hispec*/cals/KOA_CALIBRATION_CACHE and HISPEC_KOA_CACHE_SEED).
# KOA_SEED_SOURCES = ["/Users/ckilpatrick/hispec_rv_recovery_sim/cals/KOA_CALIBRATION_CACHE"]
KOA_SEED_SOURCES = None

summary = ensure_koa_calibrations(
    SPECTROGRAPH,
    seed_sources=KOA_SEED_SOURCES,
    ignore_missing_cal_types=["lfc_wavesol"],
)

seed = summary.get("seed") or {}
if seed.get("seeded"):
    print(f"seeded KOA cache from {seed['source']}  ({seed['n_fits_before']} -> {seed['n_fits_after']} FITS)")
elif seed.get("n_fits_before", 0):
    print(f"KOA cache already populated ({seed['n_fits_before']} FITS)")
print(f"remote_mode: {summary.get('remote_mode')}")
if summary.get("remote_error"):
    print(f"remote_error: {summary['remote_error']}")

print(
    f"{summary['spectrograph']} caldb: {summary['n_records']} record(s); "
    f"downloaded {summary['n_downloaded']}/{summary['n_missing']} missing file(s)"
)
print("types:", summary["cal_types"])
if summary["downloaded"]:
    print("downloaded:", summary["downloaded"])
if summary["failed"]:
    print("failed:", summary["failed"])
if summary.get("still_missing"):
    print("still_missing:", summary["still_missing"])


## 2. Injection / scan configuration

Fiber 2. Fixed Aidan WAVE + deg-0 order ZP. Spectrum Doppler paint only.
Near-noiseless UTR. Packaged static `ccfopt.delta.mas` with line-by-line stellar RV.
Scan: injected RV = 0, 100, …, 1000 m/s (11 steps).

**Cache flags** (defaults avoid redoing expensive work):

| Flag | Default | Effect |
|---|---|---|
| `RESIMULATE_RAW` | `False` | Re-run 1D→ZEMAX→UTR only if True or ramp missing |
| `RERUN_PIPELINE` | `False` | Re-run Stage0–2 (through blaze, **no** stellar RV) only if True or blaze L2 missing |
| `RERUN_STELLAR_RV` | `False` | Re-run stellar RV on all injects only if True or RVCCF missing |
| `REBUILD_SHARED_CALS` | `False` | Rebuild shared Stage-1 bundle only if True or missing |

After changing the mask or stellar-RV / CCF code, set `RERUN_STELLAR_RV=True` and
re-run **section 6** only.


In [ ]:
from pathlib import Path

import numpy as np

from hispecdrp import datamodels
from hispecdrp.utils.line_mask_utils import (
    CCFOPT_SINGLE_LINE_STELLAR_RV_KW,
    packaged_default_line_mask_path,
)

FIBERS = [2]
INJECT_RV_VIA = "spectrum"
EXTRACTION_APERTURE = 8
TEMPLATE = "lte049.0-2.5-0.0a+0.0.BT-Settl.spec.7.h5"
MASK_PATH = packaged_default_line_mask_path()

POISSON_NOISE = False
CONVERT_TO_UINT16 = False
SEED = 0

# Cache / rerun controls — leave False unless you intentionally want a rebuild.
RESIMULATE_RAW = False
RERUN_PIPELINE = False  # Stage0–2 through blaze only (stellar_rv skipped)
RERUN_STELLAR_RV = False  # set True to refresh RV on all injects after mask/CCF changes
REBUILD_SHARED_CALS = False

# 0 → 1000 m/s in 100 m/s steps (11 runs)
INJECT_M_S = np.arange(0.0, 1000.0 + 0.5, 100.0)
assert len(INJECT_M_S) == 11


def inj_label(inject_m_s: float) -> str:
    return f"inj_{int(round(float(inject_m_s))):05d}mps"


def inj_dir(inject_m_s: float) -> Path:
    return Path(OUT) / inj_label(inject_m_s)


def ramp_path(inject_m_s: float) -> Path:
    return inj_dir(inject_m_s) / "simulated_ramp.fits"


def find_l2_blaze_path(run_dir: Path | str) -> Path | None:
    """Latest blaze-corrected L2 (`HB.*_spec*.fits`, not uncal). May lack RVCCF."""
    run_dir = Path(run_dir)
    cands = [
        p
        for p in sorted(run_dir.glob("HB.*_spec*.fits"))
        if "uncal" not in p.name
    ]
    return cands[-1] if cands else None


def find_l2_path(run_dir: Path | str) -> Path | None:
    """Prefer stellar_rv product; else latest blaze L2 that already has RVCCF."""
    run_dir = Path(run_dir)
    hits = sorted(run_dir.glob("HB.*_spec-stellar_rv*.fits"))
    if hits:
        return hits[-1]
    for p in reversed(list(run_dir.glob("HB.*_spec*.fits"))):
        if "uncal" in p.name:
            continue
        if l2_has_rvccf(p):
            return p
    return None


def l2_has_rvccf(path: Path | str | None) -> bool:
    if path is None or not Path(path).is_file():
        return False
    try:
        m = datamodels.open(str(path))
    except Exception:
        return False
    tbl = getattr(m, f"rvccf_fib{FIBERS[0]}", None)
    return tbl is not None and len(tbl) > 0


def stellar_rv_prim_args(ccf_input_velocity_m_s: float, *, save_preview: bool = False) -> dict:
    kw = CCFOPT_SINGLE_LINE_STELLAR_RV_KW
    return {
        "algorithm": "ccf_line_mask",
        "mask_path": MASK_PATH,
        "ccf_input_velocity": float(ccf_input_velocity_m_s),
        "do_fibers": list(FIBERS),
        "window_line_by_line": bool(kw["window_line_by_line"]),
        "window_size_um": float(kw["window_size_um"]),
        "v_half_window": float(kw["v_half_window_m_s"]),
        "v_step_coarse": float(kw["v_step_m_s"]),
        "mask_auto_center_scout": bool(kw["mask_auto_center_scout"]),
        "mask_ccf_pixel_interp_order": int(kw["mask_ccf_pixel_interp_order"]),
        "ccf_zb": 0.0,
        "ccf_zb_from_barycorr": False,
        "save_preview": bool(save_preview),
        "preview_write_png": False,
        "show_preview": False,
    }


def per_order_rv_from_l2(l2, fiber: int) -> dict[int, dict[str, float]]:
    """Inverse-variance mean RV per echelle order from RVCCF_FIB* chunks/lines."""
    tbl = getattr(l2, f"rvccf_fib{fiber}", None)
    if tbl is None or len(tbl) == 0:
        raise RuntimeError(f"missing rvccf_fib{fiber}")
    orders = np.asarray(tbl["ORDER"], dtype=int)
    rvs = np.asarray(tbl["RV"], dtype=float)
    errs = np.asarray(tbl["RV_ERR"], dtype=float)
    if "INLIER" in tbl.colnames:
        inl = np.asarray(tbl["INLIER"], dtype=float) > 0
    else:
        inl = np.ones(len(tbl), dtype=bool)
    out: dict[int, dict[str, float]] = {}
    for od in np.unique(orders):
        m = (orders == int(od)) & inl & np.isfinite(rvs) & np.isfinite(errs) & (errs > 0)
        if not np.any(m):
            m = (orders == int(od)) & np.isfinite(rvs) & np.isfinite(errs) & (errs > 0)
        if not np.any(m):
            continue
        w = 1.0 / np.maximum(errs[m], 1e-30) ** 2
        n_lines = int(m.sum())  # line-by-line: one RVCCF row per mask line
        out[int(od)] = {
            "rv_m_s": float(np.average(rvs[m], weights=w)),
            "rv_err_m_s": float(np.sqrt(1.0 / np.sum(w))),
            "n_lines": n_lines,
            "n_chunks": n_lines,
            "median_m_s": float(np.median(rvs[m])),
        }
    return out


print(f"fiber(s): {FIBERS}")
print(f"extraction_aperture = {EXTRACTION_APERTURE}")
print(f"inject_rv_via = {INJECT_RV_VIA!r}")
print(f"poisson_noise={POISSON_NOISE}  convert_to_uint16={CONVERT_TO_UINT16}")
print(f"mask_path = {MASK_PATH}")
print(f"seed={SEED}")
print(
    f"RESIMULATE_RAW={RESIMULATE_RAW}  RERUN_PIPELINE={RERUN_PIPELINE}  "
    f"RERUN_STELLAR_RV={RERUN_STELLAR_RV}  REBUILD_SHARED_CALS={REBUILD_SHARED_CALS}"
)
print(f"inject steps (m/s): {INJECT_M_S.tolist()}")


## 3. Shared Stage-1 cals

Build once at RV=0; reuse fiberflat / traces / WAVE for every scan step.
With `REBUILD_SHARED_CALS=False`, reload masters from `_shared_stage1_rv0/cals`
when present.


In [ ]:
from hispecdrp.simulations.e2e_pipeline import build_wavesol_aidan_with_order_zp
from hispecdrp.simulations.simulate_up_the_ramp import (
    create_ramp_from_spectrum,
    get_expected_ramp_path,
    load_cached_sim_stage1_masters,
    sim_stage1_masters_exist,
)

_shared_dir = str(Path(OUT) / "_shared_stage1_rv0")
os.makedirs(_shared_dir, exist_ok=True)

if (
    not REBUILD_SHARED_CALS
    and sim_stage1_masters_exist(
        _shared_dir,
        SPECTROGRAPH,
        FIBERS,
        extraction_aperture=float(EXTRACTION_APERTURE),
    )
    and Path(_shared_dir, "rate_map.fits").is_file()
):
    print(f"Reusing shared Stage-1 cals under {_shared_dir}/cals ...")
    _bundle = load_cached_sim_stage1_masters(_shared_dir, SPECTROGRAPH)
    SHARED_TRACEPOS = _bundle["trace_pos"]
    SHARED_TRACEPROF = _bundle["trace_prof"]
    SHARED_FIBERFLAT = _bundle["fiberflat"]
    from astropy.io import fits as _fits

    RATE_MAP_REF = np.asarray(_fits.getdata(str(Path(_shared_dir) / "rate_map.fits")), dtype=np.float64)
    SHARED_WAVESOL, ZP_INFO = build_wavesol_aidan_with_order_zp(
        RATE_MAP_REF,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        known_rv_m_s=0.0,
    )
else:
    print(
        "Building shared Stage-1 cals (RV=0 → fiberflat/traces aperture="
        f"{EXTRACTION_APERTURE}, Aidan gws + order ZP₀) ..."
    )
    _ramp0, _cals0, RATE_MAP_REF = create_ramp_from_spectrum(
        _shared_dir,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename=TEMPLATE,
        mag_ab=3.05,
        filter_band="J",
        seed=SEED,
        radial_velocity_km_s=0.0,
        barycentric_velocity_km_s=0.0,
        inject_rv_via=INJECT_RV_VIA,
        poisson_noise=POISSON_NOISE,
        convert_to_uint16=CONVERT_TO_UINT16,
        extraction_aperture=float(EXTRACTION_APERTURE),
        reuse_stage1_sims=not REBUILD_SHARED_CALS,
    )
    SHARED_TRACEPOS = _cals0["trace_pos"]
    SHARED_TRACEPROF = _cals0["trace_prof"]
    SHARED_FIBERFLAT = _cals0["fiberflat"]
    SHARED_WAVESOL, ZP_INFO = build_wavesol_aidan_with_order_zp(
        RATE_MAP_REF,
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        known_rv_m_s=0.0,
        meta={"mjd_start": getattr(getattr(_ramp0, "meta", None), "mjd_start", 0.0)},
    )

print(
    f"SHARED_WAVESOL note={getattr(SHARED_WAVESOL.meta, 'note', None)}  "
    f"median order ZP = {ZP_INFO.get('median_dv_m_s', float('nan')):.3f} m/s  "
    f"(poly_deg={ZP_INFO.get('poly_deg')})"
)


## 4. Simulate all raw ramps (1D → Doppler → ZEMAX → UTR)

Writes `OUT/inj_XXXXXmps/simulated_ramp.fits` (+ `rate_map.fits`).
Default `RESIMULATE_RAW=False` skips steps that already have a ramp on disk.


In [ ]:
RAW_MANIFEST = []
for inj in INJECT_M_S:
    label = inj_label(inj)
    run_dir = inj_dir(inj)
    run_dir.mkdir(parents=True, exist_ok=True)
    rpath = ramp_path(inj)
    if (not RESIMULATE_RAW) and rpath.is_file():
        print(f"[raw skip] {label}: reuse {rpath}")
        RAW_MANIFEST.append(
            {"label": label, "inject_m_s": float(inj), "ramp_path": str(rpath), "reused": True}
        )
        continue

    print(f"\n=== RAW {label}: inject {inj:.1f} m/s  (1D→ZEMAX→UTR) ===")
    ramp, _cals, rate_map = create_ramp_from_spectrum(
        str(run_dir),
        spectrograph=SPECTROGRAPH,
        fibers=FIBERS,
        template_filename=TEMPLATE,
        mag_ab=3.05,
        filter_band="J",
        seed=SEED,
        radial_velocity_km_s=float(inj) / 1000.0,
        barycentric_velocity_km_s=0.0,
        inject_rv_via=INJECT_RV_VIA,
        poisson_noise=POISSON_NOISE,
        convert_to_uint16=CONVERT_TO_UINT16,
        register_stage1_sims=False,
        overwrite_ramp=True,
    )
    print(f"  wrote {get_expected_ramp_path(str(run_dir))}  shape={ramp.data.shape}")
    RAW_MANIFEST.append(
        {
            "label": label,
            "inject_m_s": float(inj),
            "ramp_path": get_expected_ramp_path(str(run_dir)),
            "reused": False,
        }
    )

n_reuse = sum(1 for r in RAW_MANIFEST if r["reused"])
print(f"\nRaw sim done: {len(RAW_MANIFEST)} steps  ({n_reuse} reused, {len(RAW_MANIFEST) - n_reuse} rebuilt)")


## 5. Stage0–2 pipeline on cached ramps (through blaze; **no** stellar RV)

Loads each `simulated_ramp.fits`, runs Stage0→1→2 with shared cals, but sets
`stellar_rv.run=False`. Saves the blaze-corrected L2 (`HB.*_spec*.fits`).
Default `RERUN_PIPELINE=False` reuses existing blaze L2 files.

Stellar RV is **section 6** so mask / CCF changes can be re-run without Stage0–1.


In [ ]:
from hispecdrp.pipelines import Stage0Pipeline, Stage1Pipeline, Stage2Pipeline


def run_pipeline_through_blaze(inject_m_s: float) -> dict:
    """Stage0–2 through blaze (stellar_rv skipped) on cached ramp."""
    label = inj_label(inject_m_s)
    run_dir = inj_dir(inject_m_s)
    injected_m_s = float(inject_m_s)
    rpath = ramp_path(inject_m_s)
    if not rpath.is_file():
        raise FileNotFoundError(
            f"Missing raw ramp for {label}: {rpath}. Run section 4 with RESIMULATE_RAW=True."
        )

    existing = find_l2_blaze_path(run_dir)
    if (not RERUN_PIPELINE) and existing is not None and existing.is_file():
        print(f"[pipe skip] {label}: reuse blaze L2 {existing}")
        return {
            "label": label,
            "run_dir": str(run_dir),
            "l2_blaze_path": str(existing),
            "inject_m_s": injected_m_s,
        }

    print(f"\n=== PIPE {label}: inject {injected_m_s:.1f} m/s  Stage0–2 (no stellar RV) ===")
    ramp = datamodels.open(str(rpath))
    do_fibers = [str(f) for f in FIBERS]
    l0 = Stage0Pipeline(output_dir=str(run_dir)).apply(ramp)
    l1 = Stage1Pipeline(
        output_dir=str(run_dir),
        prim_args={
            "spec_extract": {
                "do_fibers": do_fibers,
                "method": "box",
                "extraction_aperture": int(EXTRACTION_APERTURE),
                "tracepos": SHARED_TRACEPOS,
                "traceprof": SHARED_TRACEPROF,
            },
            "set_wavesol": {"wavesol": SHARED_WAVESOL},
        },
    ).apply(l0)
    l2 = Stage2Pipeline(
        output_dir=str(run_dir),
        prim_args={
            "blaze_corr": {
                "do_fibers": do_fibers,
                "fiberflat": SHARED_FIBERFLAT,
            },
            "sky_sub": {"run": False},
            "barycorr": {"run": False},
            "stellar_rv": {"run": False},
        },
    ).apply(l1)
    l2_path = getattr(l2.meta, "filename", None) or str(find_l2_blaze_path(run_dir))
    save_as = run_dir / Path(l2_path).name if l2_path else run_dir / f"{label}_spec.fits"
    if not Path(l2_path).is_file():
        l2.save(filepath=str(save_as))
        l2_path = str(save_as)
    print(f"  blaze L2 saved: {l2_path}")
    return {
        "label": label,
        "run_dir": str(run_dir),
        "l2_blaze_path": str(l2_path),
        "inject_m_s": injected_m_s,
    }


PIPE_RESULTS = []
for inj in INJECT_M_S:
    PIPE_RESULTS.append(run_pipeline_through_blaze(inj))

print(f"\nBlaze L2 products ({len(PIPE_RESULTS)}):")
for r in PIPE_RESULTS:
    print(f"  {r['label']}: {r['l2_blaze_path']}")


## 6. Stellar RV on blaze L2 products

Runs `StellarRVPrimitive` on each inject's blaze-corrected L2, writes
`RVCCF_FIB*` / `RVCCFAVG_FIB*`, and rebuilds `per_order_rv_scan.ecsv`.

Set `RERUN_STELLAR_RV=True` (section 2) and re-run **only this cell** after
updating the mask or stellar-RV / CCF code — no need to re-simulate or re-run
Stage0–1.


In [ ]:
from astropy.table import Table

from hispecdrp.primitives.stellar_rv.stellar_rv_primitive import StellarRVPrimitive


def run_stellar_rv_on_inject(inject_m_s: float, *, save_preview: bool = False) -> dict:
    """Stellar RV only on an existing blaze L2; return aggregate / per-order RVs."""
    label = inj_label(inject_m_s)
    run_dir = inj_dir(inject_m_s)
    injected_m_s = float(inject_m_s)

    blaze = find_l2_blaze_path(run_dir)
    if blaze is None or not blaze.is_file():
        raise FileNotFoundError(
            f"Missing blaze L2 for {label} under {run_dir}. Run section 5 first."
        )

    existing_rv = find_l2_path(run_dir)
    if (not RERUN_STELLAR_RV) and l2_has_rvccf(existing_rv):
        print(f"[rv skip] {label}: reuse RVCCF in {existing_rv}")
        l2 = datamodels.open(str(existing_rv))
        l2_path = str(existing_rv)
    else:
        print(
            f"\n=== STELLAR RV {label}: inject {injected_m_s:.1f} m/s  "
            f"input={blaze.name} ==="
        )
        prim = StellarRVPrimitive(
            **stellar_rv_prim_args(injected_m_s, save_preview=bool(save_preview)),
            save_result=True,
            output_dir=str(run_dir),
            suffix="stellar_rv",
        )
        l2 = prim.apply(str(blaze))
        l2_path = getattr(l2.meta, "filename", None) or str(find_l2_path(run_dir) or blaze)
        if not Path(l2_path).is_file():
            save_as = run_dir / f"{Path(blaze).stem}-stellar_rv.fits"
            l2.save(filepath=str(save_as))
            l2_path = str(save_as)
        print(f"  L2+RV saved: {l2_path}")

    fib = FIBERS[0]
    avg = getattr(l2, f"rvccfavg_fib{fib}", None)
    if avg is None or len(avg) == 0:
        raise RuntimeError(f"{label}: missing rvccfavg_fib{fib} in {l2_path}")
    rv_rec = float(avg["RV"][0])
    rv_err = float(avg["RV_ERR"][0])
    per_order = per_order_rv_from_l2(l2, fib)
    print(
        f"  fiber {fib}: agg RV = {rv_rec:.6f} ± {rv_err:.6f} m/s  "
        f"bias = {rv_rec - injected_m_s:.6f} m/s  n_orders={len(per_order)}"
    )
    return {
        "label": label,
        "run_dir": str(run_dir),
        "l2_path": str(l2_path),
        "inject_m_s": injected_m_s,
        "recover_m_s": rv_rec,
        "recover_err_m_s": rv_err,
        "per_order": per_order,
    }


RESULTS = []
for inj in INJECT_M_S:
    RESULTS.append(run_stellar_rv_on_inject(inj, save_preview=False))

rows = []
for r in RESULTS:
    for od, d in sorted(r["per_order"].items()):
        rows.append(
            {
                "label": r["label"],
                "inject_m_s": r["inject_m_s"],
                "order": int(od),
                "rv_order_m_s": d["rv_m_s"],
                "rv_order_err_m_s": d["rv_err_m_s"],
                "rv_order_median_m_s": d["median_m_s"],
                "n_lines": int(d["n_lines"]),
                "n_chunks": int(d["n_chunks"]),
                "bias_order_m_s": d["rv_m_s"] - r["inject_m_s"],
                "rv_agg_m_s": r["recover_m_s"],
                "bias_agg_m_s": r["recover_m_s"] - r["inject_m_s"],
                "l2_path": r["l2_path"],
            }
        )
ORDER_TBL = Table(rows)
order_csv = Path(OUT) / "per_order_rv_scan.ecsv"
ORDER_TBL.write(order_csv, format="ascii.ecsv", overwrite=True)
print(f"\nWrote {order_csv}  ({len(ORDER_TBL)} rows)")
print("\nAggregate recovery:")
for r in RESULTS:
    print(
        f"  inject={r['inject_m_s']:7.1f} m/s  "
        f"agg={r['recover_m_s']:+10.3f}  bias={r['recover_m_s'] - r['inject_m_s']:+8.3f} m/s  "
        f"L2={Path(r['l2_path']).name}"
    )


## 7. Plot: injected vs recovered RV (order by order)

Reads **only** `per_order_rv_scan.ecsv` (and optionally L2 paths listed there).
Re-run this cell after changing plot style without re-simulating or re-pipelining.

Bias heatmap: each cell shows the **number of lines** used for that
(order, inject) RV. When `n_lines` changes between adjacent inject steps in the
same order, the shared cell border is drawn in **bold gold**.

Plots via `hispecdrp.simulations.plotting.write_per_order_rv_scan_report`.


In [ ]:
from hispecdrp.simulations.plotting import write_per_order_rv_scan_report

fib = FIBERS[0]
summary = write_per_order_rv_scan_report(
    OUT,
    fiber=fib,
    spectrograph=SPECTROGRAPH,
    show=True,
)
ORDER_TBL = summary["order_table"]
print(f"plotting n_inject={summary['n_inject']}  n_orders={summary['n_orders']}  (n_col={summary['n_col']})")
print(f"wrote {summary['pdf']}")
print(f"wrote {summary['png']}")
print(
    f"bias stats: median={summary['bias_median_m_s']:+.3f} m/s  "
    f"MAD={summary['bias_mad_m_s']:.3f} m/s  "
    f"|max|={summary['bias_abs_max_m_s']:.3f} m/s"
)
print(f"n_lines changes between adjacent inject steps: {summary['n_line_count_changes']}")
